In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import geopandas as gpd
import geobr
import seaborn as sns
import scipy.stats as ss
sns.set(style="ticks")
import lovelyplots
plt.style.use('ipynb')

In [ ]:
sys.path.append('..')

## Loading cases by municipality

In [ ]:
df_sp = pd.read_csv('data/municipalities_SP.csv')

## Processing data

In [ ]:
df_muns = pd.read_csv('data/age_groups_municipalities.csv')

In [ ]:
df_muns['Estado'] = df_muns['Municipio'].astype(str).str[:2]

In [ ]:
df_muns = df_muns[df_muns['Estado'] == '35']

In [ ]:
df_muns = df_muns[['Municipio','Total', 'Estado']]

## Including municipality population

In [ ]:
df_sp = df_sp.merge(df_muns, how = 'outer',left_on = 'ID_MN_RESI', right_on = 'Municipio')

In [ ]:
df_sp = df_sp.drop(columns = ['Unnamed: 0','ID_MN_RESI'])

In [ ]:
df_sp = df_sp.fillna(0)

In [ ]:
df_sp['CASO_N'] = df_sp['CASO']/df_sp['Total']

## Maps of SP

In [ ]:
map_br = geobr.read_municipality()

In [ ]:
df_sp.loc[df_sp['CASO'] == 0,'COV_1'] = np.nan

In [ ]:
map_br['code_muni'] = map_br['code_muni'].astype(str).str[:6].astype(int)
map_br = map_br.merge(df_sp, how = 'left', left_on = 'code_muni', right_on = 'Municipio')

In [ ]:
map_br = map_br[map_br['abbrev_state'] == 'SP']

In [ ]:
map_br['CASO_100'] = map_br['CASO_N']*(10**5)

In [ ]:
map_br['CASO_100_LOG'] = np.log(map_br['CASO_100'])

In [ ]:
missing_kwds = dict(color='grey', label='No Data')

fig, axs = plt.subplots(figsize = (12,12))

map_br.plot(
        ax=axs, 
        column='CASO_100', 
        missing_kwds=missing_kwds,
        linewidth=0.25, 
        cmap = 'viridis'
    )


cax = fig.add_axes(
        [
            0.9,    # posicao x (entre 0.0 e 1.0)
            0.18,    # posicao y (entre 0.0 e 1.0)
            0.03,    # largura x
            0.40,    # altura y
        ]
    )

sm = plt.cm.ScalarMappable(
        cmap="viridis",                                       
        norm=plt.Normalize(
            vmin=map_br["CASO_100"].min(),  
            vmax=map_br["CASO_100"].max(),  
        ),
    )

cbar = fig.colorbar(
    sm,
    cax=cax)

cbar.set_label("Incidence per 100.000 inhabitants", fontsize=12, rotation=270, labelpad=20)

axs.axis("off")
plt.savefig('figures/incidence_sp.pdf')
plt.show()

In [ ]:
missing_kwds = dict(color='grey', label='No Data')

fig, axs = plt.subplots(figsize = (12,12))

map_br.plot(
        ax=axs, 
        column='COV_1', 
        missing_kwds=missing_kwds,
        linewidth=0.25,
        edgecolor = 'black',
        cmap = 'Greens'
    )


cax = fig.add_axes(
        [
            0.9,    # posicao x (entre 0.0 e 1.0)
            0.18,    # posicao y (entre 0.0 e 1.0)
            0.03,    # largura x
            0.40,    # altura y
        ]
    )

sm = plt.cm.ScalarMappable(
        cmap="Greens",                                       
        norm=plt.Normalize(
            vmin=map_br["COV_1"].min(),  
            vmax=map_br["COV_1"].max(),  
        ),
    )
    
cbar= fig.colorbar(
    sm,
    cax=cax,
    )

cbar.set_label("Estimated vaccine coverage", fontsize=12, rotation=270, labelpad=20)

#axs.set_title(
#    "Cases per 100k in the SP state",
#    fontdict={"fontsize": 15},
#    )

axs.axis("off")
plt.savefig('figures/coverage_sp.pdf')
plt.show()